# 02. Model Testing & Evaluation

**Goal:**  
Evaluate the generalization performance of our custom Multinomial Naive Bayes model on the unseen validation dataset.

**Pipeline Overview:**
1. **Data Ingestion:** Load the preprocessed validation dataset.
2. **Artifact Loading:** Load the saved `CountVectorizer` and fitted `MultinomialNaiveBayes` model from disk.
3. **Strict Transformation:** Apply the vectorizer's vocabulary to the new data using `.transform()` (preventing data leakage).
4. **Inference:** Generate sentiment predictions.
5. **Evaluation:** Compute the confusion matrix, accuracy, precision, recall, and Macro-F1 score strictly using our custom math in `src/metrics.py`.

In [ ]:
import sys
import os
import pickle
import pandas as pd
import numpy as np

# Ensure Python can locate modules inside the src/ directory
sys.path.append(os.path.abspath("../src"))

from preprocessing import transform_new_data
from naive_bayes import MultinomialNaiveBayes
from metrics import (compute_classification_metrics, plot_confusion_matrix)

# 1. Load the pre-cleaned validation data 
val_data_path = "../data/processed/validation_cleaned.csv"
df_val = pd.read_csv(val_data_path)

# Drop any nulls that might occur if a tweet became purely empty whitespace after cleaning
df_val = df_val.dropna(subset=['tweet_content'])

print(f"Loaded validation samples: {len(df_val)}")
print("\nValidation Class Distribution:")
print(df_val['sentiment'].value_counts())

ImportError: cannot import name 'plot_confusion_matrix' from 'metrics' (c:\Users\adoni\OneDrive\Desktop\iCog Training\Training 1\naive-bayes-from-scratch\src\metrics.py)

## 2. Load Pre-trained Artifacts

We load the vectorizer and the model saved at the end of the training phase. By using a pre-fitted vectorizer, we guarantee that the sparse feature matrix generated for the validation set matches the exact column dimensions and vocabulary learned during training.

In [5]:
models_dir = "../models"

# Load the vectorizer
vectorizer_path = os.path.join(models_dir, "vectorizer.pkl")
with open(vectorizer_path, "rb") as f:
    vectorizer = pickle.load(f)

# Load the model
model_path = os.path.join(models_dir, "naive_bayes_model.pkl")
model = MultinomialNaiveBayes.load_model(model_path)

print(f"Successfully loaded vectorizer (Vocabulary size: {len(vectorizer.vocabulary_)})")
print(f"Successfully loaded model (Learned classes: {model.classes_})")

Successfully loaded vectorizer (Vocabulary size: 14250)
Successfully loaded model (Learned classes: ['Negative' 'Neutral' 'Positive'])


## 3. Strict Transformation

We isolate the features (`X`) and targets (`y`). Then, we pass the text through `transform_new_data`.
**Crucial step:** We only call `.transform()` here. Calling `.fit()` or `.fit_transform()` on validation data is a critical data leakage error that overwrites the training vocabulary.

In [6]:
X_val_text = df_val['tweet_content']
y_true = df_val['sentiment'].values

# Transform validation text to sparse matrix using the fitted vectorizer
X_val_sparse = transform_new_data(vectorizer, X_val_text)

print(f"Validation feature matrix shape: {X_val_sparse.shape}")

Validation feature matrix shape: (827, 14250)


## 4. Inference

We pass the sparse validation matrix into our model's `predict()` method. The model computes the log-posterior score for each class via fast matrix multiplication and returns the class with the highest score.

In [8]:
# Generate predictions
y_pred = model.predict(X_val_sparse)

# Show a quick sanity check of the first 5 predictions vs actuals
print("First 5 Predictions vs Actuals:")
for i in range(10):
    print(f"Predicted: {y_pred[i]:<10} | Actual: {y_true[i]}")

First 5 Predictions vs Actuals:
Predicted: Neutral    | Actual: Neutral
Predicted: Negative   | Actual: Negative
Predicted: Negative   | Actual: Negative
Predicted: Neutral    | Actual: Neutral
Predicted: Negative   | Actual: Negative
Predicted: Positive   | Actual: Positive
Predicted: Positive   | Actual: Positive
Predicted: Positive   | Actual: Positive
Predicted: Negative   | Actual: Negative
Predicted: Positive   | Actual: Positive


## 5. Model Evaluation

We evaluate the predictions using our custom `src/metrics.py` functions. The Macro-F1 score is our primary north-star metric, ensuring the model performs well across all classes evenly. The confusion matrix helps us diagnose specific classification overlaps.

In [10]:
# Compute all metrics from scratch
results = compute_classification_metrics(y_true, y_pred, classes=model.classes_)

# 1. Print Overall Summary
print("="*40)
print(f"OVERALL ACCURACY:  {results['accuracy']:.4f}")
print(f"MACRO F1-SCORE:    {results['macro']['macro_f1']:.4f}")
print("="*40)

# 2. Print Per-Class Metrics
print("\n--- PER-CLASS METRICS ---")
for c in results['classes']:
    metrics = results['per_class'][c]
    print(f"Class: {c}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1-Score:  {metrics['f1_score']:.4f}")
    print(f"  Support:   {metrics['support']}")
    print("-" * 25)

# 3. Print Confusion Matrix
print("\n--- CONFUSION MATRIX ---")
print(f"Rows: Actual | Columns: Predicted")
print(f"Labels: {results['classes']}")
print(results['confusion_matrix'])

#to Visualize the confusion matrix
plot_confusion_matrix(
    cm=results['confusion_matrix'], 
    classes=results['classes'], 
    save_path="../models/confusion_matrix.png"
)



OVERALL ACCURACY:  0.8174
MACRO F1-SCORE:    0.8161

--- PER-CLASS METRICS ---
Class: Negative
  Precision: 0.7857
  Recall:    0.8684
  F1-Score:  0.8250
  Support:   266
-------------------------
Class: Neutral
  Precision: 0.9005
  Recall:    0.6982
  F1-Score:  0.7866
  Support:   285
-------------------------
Class: Positive
  Precision: 0.7885
  Recall:    0.8913
  F1-Score:  0.8367
  Support:   276
-------------------------

--- CONFUSION MATRIX ---
Rows: Actual | Columns: Predicted
Labels: ['Negative' 'Neutral' 'Positive']
[[231  11  24]
 [ 44 199  42]
 [ 19  11 246]]


NameError: name 'plot_confusion_matrix' is not defined